# 03 — Embedding Exploration: Trực Quan Hoá Không Gian Embedding

**Vai trò:** Data Engineer · **Task:** S4-DE-01 (Yêu cầu 9.6)

`OllamaEmbeddingModel.embed_text()`/`embed_batch()` (S2-ME-01, S2-ME-02) chuyển mỗi đoạn văn bản thành một vector hàng trăm chiều (`nomic-embed-text` → 768 chiều). Con người không thể "nhìn" trực tiếp một không gian 768 chiều — nhưng bằng cách **giảm chiều** (PCA, t-SNE) xuống còn 2D, ta có thể trực quan hoá và quan sát một tính chất cốt lõi của embedding tốt: **các đoạn văn bản có nghĩa gần nhau thường nằm gần nhau trong không gian vector**.

Notebook này nhúng (embed) các chunk từ **hai chủ đề khác biệt rõ rệt**, giảm chiều bằng PCA và t-SNE, rồi vẽ biểu đồ tương tác bằng `plotly` (Yêu cầu 9.6).

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.models import ChunkStrategy
from src.pipeline.experiment_tracker import ExperimentTracker

print(f"Project root: {PROJECT_ROOT}")

## 1. Chuẩn bị: hai tài liệu mẫu thuộc hai chủ đề khác biệt

Để biểu đồ giảm chiều thể hiện rõ "các đoạn nghĩa gần nhau nằm gần nhau", ta cần các chunk thuộc **nhiều chủ đề khác biệt**. Notebook tạo (idempotent — chạy lại không nhân bản, Yêu cầu 9.1) hai tài liệu: một về RAG/AI, một về ẩm thực — rồi chia nhỏ thành chunk ngắn (`chunk_size=120`) để có đủ điểm dữ liệu cho biểu đồ.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

sample_docs = {
    "embedding_topic_rag.txt": (
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop retrieval va "
        "generation. He thong tim cac doan van ban lien quan tu kho du lieu rieng "
        "truoc khi yeu cau LLM sinh cau tra loi, giup giam hien tuong ao giac va "
        "bam sat nguon tai lieu thuc te. Embedding model chuyen van ban thanh "
        "vector so, cho phep so sanh ngu nghia bang khoang cach trong khong gian "
        "vector. Vector store nhu ChromaDB luu tru cac vector nay va ho tro tim "
        "kiem tuong dong (similarity search) de truy xuat ngu canh lien quan "
        "truoc khi sinh cau tra loi."
    ),
    "embedding_topic_cooking.txt": (
        "Pho la mon an truyen thong noi tieng cua Viet Nam, gom banh pho, nuoc "
        "dung ham tu xuong va thit bo hoac ga, an kem rau thom va gia. Nuoc dung "
        "ngon doi hoi ham xuong trong nhieu gio voi gung nuong va cac loai gia vi "
        "nhu hoi, que, thao qua de tao huong thom dac trung. Banh cuon, bun cha "
        "va com tam cung la nhung mon an pho bien khac the hien su da dang cua "
        "am thuc duong pho Viet Nam, thuong duoc thuong thuc vao buoi sang som."
    ),
}

for name, content in sample_docs.items():
    path = RAW_DIR / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")

loader = DocumentLoader()
chunker = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=120, chunk_overlap=20)

all_chunks = []
chunk_topics = []
for name in sample_docs:
    document = loader.load(str(RAW_DIR / name))
    doc_chunks = chunker.chunk(document)
    topic = "rag_ai" if "rag" in name else "am_thuc"
    all_chunks.extend(doc_chunks)
    chunk_topics.extend([topic] * len(doc_chunks))

print(f"Tong so chunk: {len(all_chunks)}")
print(f"  - rag_ai : {chunk_topics.count('rag_ai')} chunk")
print(f"  - am_thuc: {chunk_topics.count('am_thuc')} chunk")

## 2. Tạo embedding thật qua `embed_batch()`

`embed_batch()` đảm bảo `embed_batch(texts)[i] == embed_text(texts[i])` (Property 5, S2-ME-02) — ở đây ta dùng nó để nhúng toàn bộ các chunk cùng lúc và đo thời gian xử lý.

In [ ]:
embedder = OllamaEmbeddingModel(model_name="nomic-embed-text")

embed_start = time.perf_counter()
vectors = embedder.embed_batch([c.content for c in all_chunks])
embed_latency_ms = (time.perf_counter() - embed_start) * 1000

vectors_np = np.array(vectors)
assert len(vectors) == len(all_chunks)
assert vectors_np.shape[1] == embedder.dimension

print(f"So vector       : {vectors_np.shape[0]}")
print(f"Chieu embedding : {vectors_np.shape[1]} (model = {embedder.model_name})")
print(f"Thoi gian embed_batch: {embed_latency_ms:.1f} ms")

## 3. Giảm chiều bằng PCA — chiếu tuyến tính lên 2 trục phương sai lớn nhất

PCA (Principal Component Analysis) tìm 2 hướng giữ lại nhiều phương sai nhất của dữ liệu gốc và chiếu các vector lên đó. Đây là phép giảm chiều **tuyến tính, nhanh, tất định** — phù hợp để có cái nhìn tổng quan đầu tiên về không gian embedding.

In [ ]:
def make_preview(text, limit=90):
    flat = " ".join(text.strip().split())
    return flat[:limit] + ("…" if len(flat) > limit else "")


previews = [make_preview(c.content) for c in all_chunks]

pca_coords = PCA(n_components=2, random_state=42).fit_transform(vectors_np)
df_pca = pd.DataFrame({
    "x": pca_coords[:, 0],
    "y": pca_coords[:, 1],
    "chu_de": chunk_topics,
    "noi_dung": previews,
})

fig_pca = px.scatter(
    df_pca, x="x", y="y", color="chu_de", hover_data=["noi_dung"],
    title=f"PCA — khong gian embedding '{embedder.model_name}' ({embedder.dimension}D → 2D)",
    labels={"x": "Thanh phan chinh 1", "y": "Thanh phan chinh 2", "chu_de": "Chu de"},
)
fig_pca.update_traces(marker=dict(size=10, opacity=0.8))
fig_pca

## 4. Giảm chiều bằng t-SNE — bảo toàn cấu trúc lân cận cục bộ

t-SNE (t-distributed Stochastic Neighbor Embedding) là phép giảm chiều **phi tuyến**, tối ưu để các điểm gần nhau trong không gian gốc vẫn gần nhau sau khi chiếu xuống 2D — thường tách cụm rõ hơn PCA với dữ liệu embedding văn bản. `perplexity` được giới hạn theo số điểm dữ liệu hiện có để tránh lỗi với tập nhỏ.

In [ ]:
perplexity = max(2, min(30, len(vectors_np) - 1))
tsne_coords = TSNE(
    n_components=2, random_state=42, perplexity=perplexity, init="pca", learning_rate="auto",
).fit_transform(vectors_np)

df_tsne = pd.DataFrame({
    "x": tsne_coords[:, 0],
    "y": tsne_coords[:, 1],
    "chu_de": chunk_topics,
    "noi_dung": previews,
})

fig_tsne = px.scatter(
    df_tsne, x="x", y="y", color="chu_de", hover_data=["noi_dung"],
    title=f"t-SNE (perplexity={perplexity}) — khong gian embedding '{embedder.model_name}'",
    labels={"x": "t-SNE chieu 1", "y": "t-SNE chieu 2", "chu_de": "Chu de"},
)
fig_tsne.update_traces(marker=dict(size=10, opacity=0.8))
fig_tsne

## 5. Ghi lại thực nghiệm qua `ExperimentTracker`

Theo Yêu cầu 9.8, `ExperimentTracker` phải có thể ghi lại kết quả thực nghiệm trong notebook **mà không làm gián đoạn luồng** — ở đây ta log lượt nhúng vừa thực hiện như một sự kiện "indexing" (S4-PE-01) rồi xem `get_summary()`.

In [ ]:
tracker = ExperimentTracker()
tracker.log_indexing(
    doc_id="embedding_exploration_batch",
    chunk_strategy=chunker.strategy.value,
    chunk_size=chunker.chunk_size,
    num_chunks=len(all_chunks),
    latency_ms=embed_latency_ms,
)

print("Tom tat phien thuc nghiem:")
print(tracker.get_summary())

## 6. Tổng kết

- `OllamaEmbeddingModel.embed_batch()` ánh xạ mỗi chunk văn bản thành một vector `{embedder.dimension}` chiều — quá nhiều để con người quan sát trực tiếp.
- **PCA** (tuyến tính, tất định, nhanh) và **t-SNE** (phi tuyến, bảo toàn lân cận cục bộ) đều giảm chiều xuống 2D để trực quan hoá bằng `plotly` (Yêu cầu 9.6) — quan sát biểu đồ ở mục 3-4, các chunk thuộc cùng chủ đề (`rag_ai` / `am_thuc`) có xu hướng tạo thành cụm riêng biệt, minh hoạ trực quan cho việc embedding "tốt" đặt các đoạn có nghĩa tương đồng gần nhau trong không gian vector — chính là nền tảng giúp `similarity_search()` (Property 6/7/8) truy xuất đúng ngữ cảnh cho `RAGPipeline.query()`.
- `ExperimentTracker.log_indexing()` ghi lại thực nghiệm vào phiên hiện tại mà không làm gián đoạn notebook (Yêu cầu 9.8).